# SB System - MT5 to PostgreSQL Import

This notebook prepares the first market-data ingestion path for SB System.

Use it in two modes:

- **Mac mode:** validate PostgreSQL connection, create schema, and test candle writes with sample data.
- **Windows VPS mode:** connect to MetaTrader 5, fetch candles from `2026-01-01` onward, and save them into PostgreSQL.

The MT5 Python package normally requires a Windows machine with MetaTrader 5 installed and logged in. Keep MT5 extraction on the Windows VPS, then point it to the same PostgreSQL database.

## 1. Load Project Helpers and Configuration

Before running this notebook, copy `.env.example` to `.env` and update `DATABASE_URL`, `SB_SYMBOLS`, `SB_TIMEFRAMES`, and `SB_IMPORT_START` as needed.

In [ ]:
from __future__ import annotations

import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sb_system.market_data import (
    check_connection,
    create_db_engine,
    create_schema,
    fetch_candle_summary,
    finish_import_run,
    load_config,
    normalize_rates,
    start_import_run,
    upsert_candles,
    upsert_symbol,
    utc_datetime_from_date,
)

config = load_config(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Platform: {platform.platform()}")
print(f"Symbols: {config.symbols}")
print(f"Timeframes: {config.timeframes}")
print(f"Import start: {config.import_start}")

## 2. Connect to PostgreSQL and Create Schema

In [ ]:
engine = create_db_engine(config.database_url)
create_schema(engine)
check_connection(engine)

## 3. Mac Validation: Insert Sample Candles

Run this cell on macOS to confirm the database schema and upsert path work before moving to the Windows VPS. It writes a tiny synthetic dataset under symbol `SBTEST`.

In [ ]:
sample = pd.DataFrame(
    [
        {
            "broker_symbol": "SBTEST",
            "timeframe": "M15",
            "candle_time": "2026-01-01T00:00:00Z",
            "open": 1.1000,
            "high": 1.1010,
            "low": 1.0990,
            "close": 1.1005,
            "tick_volume": 100,
            "spread": 12,
            "real_volume": 0,
        },
        {
            "broker_symbol": "SBTEST",
            "timeframe": "M15",
            "candle_time": "2026-01-01T00:15:00Z",
            "open": 1.1005,
            "high": 1.1020,
            "low": 1.1000,
            "close": 1.1015,
            "tick_volume": 120,
            "spread": 11,
            "real_volume": 0,
        },
    ]
)

rows = upsert_candles(engine, sample)
print(f"Upserted {rows} sample candles")
fetch_candle_summary(engine)

## 4. Windows VPS: Connect to MT5

Run this section on the Windows VPS after installing `requirements-win-mt5.txt` and opening/logging into the MetaTrader 5 terminal.

If your broker uses suffixes like `EURUSDm` or `XAUUSD.a`, set those exact names in `SB_SYMBOLS`.

In [ ]:
try:
    import MetaTrader5 as mt5
except ImportError as exc:
    raise ImportError(
        "MetaTrader5 package is not installed. On Windows VPS run: "
        "pip install -r requirements-win-mt5.txt"
    ) from exc

if not mt5.initialize():
    raise RuntimeError(f"MT5 initialize failed: {mt5.last_error()}")

account_info = mt5.account_info()
terminal_info = mt5.terminal_info()

print("MT5 initialized")
print(f"Account: {account_info.login if account_info else 'unknown'}")
print(f"Terminal path: {terminal_info.path if terminal_info else 'unknown'}")

## 5. Import MT5 Candles from 2026 Onward

In [ ]:
TIMEFRAME_MAP = {
    "M1": mt5.TIMEFRAME_M1,
    "M2": mt5.TIMEFRAME_M2,
    "M3": mt5.TIMEFRAME_M3,
    "M4": mt5.TIMEFRAME_M4,
    "M5": mt5.TIMEFRAME_M5,
    "M6": mt5.TIMEFRAME_M6,
    "M10": mt5.TIMEFRAME_M10,
    "M12": mt5.TIMEFRAME_M12,
    "M15": mt5.TIMEFRAME_M15,
    "M20": mt5.TIMEFRAME_M20,
    "M30": mt5.TIMEFRAME_M30,
    "H1": mt5.TIMEFRAME_H1,
    "H2": mt5.TIMEFRAME_H2,
    "H3": mt5.TIMEFRAME_H3,
    "H4": mt5.TIMEFRAME_H4,
    "H6": mt5.TIMEFRAME_H6,
    "H8": mt5.TIMEFRAME_H8,
    "H12": mt5.TIMEFRAME_H12,
    "D1": mt5.TIMEFRAME_D1,
    "W1": mt5.TIMEFRAME_W1,
    "MN1": mt5.TIMEFRAME_MN1,
}

date_from = utc_datetime_from_date(config.import_start)
date_to = datetime.now(timezone.utc)
total_rows = 0

import_run_id = start_import_run(
    engine,
    source="mt5",
    symbols=config.symbols,
    timeframes=config.timeframes,
    started_from=date_from,
    notes="Initial SB System historical candle import from 2026 onward",
)

try:
    for symbol in config.symbols:
        if not mt5.symbol_select(symbol, True):
            print(f"SKIP {symbol}: symbol_select failed: {mt5.last_error()}")
            continue

        info = mt5.symbol_info(symbol)
        if info:
            upsert_symbol(
                engine,
                symbol,
                description=info.description,
                digits=info.digits,
                point=info.point,
                trade_contract_size=info.trade_contract_size,
                currency_base=info.currency_base,
                currency_profit=info.currency_profit,
                currency_margin=info.currency_margin,
            )

        for timeframe in config.timeframes:
            mt5_timeframe = TIMEFRAME_MAP.get(timeframe)
            if mt5_timeframe is None:
                print(f"SKIP {symbol} {timeframe}: unsupported timeframe")
                continue

            rates = mt5.copy_rates_range(symbol, mt5_timeframe, date_from, date_to)
            candles = normalize_rates(rates, symbol=symbol, timeframe=timeframe)
            inserted = upsert_candles(engine, candles)
            total_rows += inserted
            print(f"{symbol:12} {timeframe:4} {inserted:8} candles")

    finish_import_run(engine, import_run_id, status="success", rows_imported=total_rows)
except Exception:
    finish_import_run(engine, import_run_id, status="failed", rows_imported=total_rows)
    raise
finally:
    mt5.shutdown()

print(f"Total imported rows: {total_rows}")

## 6. Verify Stored Data

In [ ]:
summary = fetch_candle_summary(engine)
summary

In [ ]:
with engine.connect() as conn:
    recent = pd.read_sql_query(
        text(
            """
            SELECT
                s.broker_symbol,
                c.timeframe,
                c.candle_time,
                c.open,
                c.high,
                c.low,
                c.close,
                c.tick_volume,
                c.spread
            FROM market.candles c
            JOIN market.symbols s ON s.symbol_id = c.symbol_id
            ORDER BY c.candle_time DESC
            LIMIT 50
            """
        ),
        conn,
    )

recent

## 7. Optional Export for Debugging

In [ ]:
export_path = PROJECT_ROOT / "data" / "exports" / "candle_summary.csv"
summary.to_csv(export_path, index=False)
print(export_path)